# ERTInv - synthetic ERT modelling & inversion with pyGIMLi

Declare your experiment in **section 2** (the only place you edit), then **Run All**.
The whole ERT workflow runs on that one description:

**true model -> forward modelling -> synthetic data -> inversion -> recovered model**

and a single PDF report is written to `results/`.


## 1. Setup
*(run once; nothing to edit here)*


In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import os, sys

# find the repo root (the folder containing 'src') so imports & outputs
# always resolve correctly, wherever this notebook lives or runs
REPO_ROOT = os.path.abspath(".")
while REPO_ROOT != os.path.dirname(REPO_ROOT) and not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    REPO_ROOT = os.path.dirname(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src import Config, Layer, Target, Survey, Forward, Inversion, run, compare_methods
print("ready. repo root:", REPO_ROOT)

ready. repo root: c:\Users\dmadelis\Documents\git\ERTInv-main


## 2. Define your experiment  <-  edit everything here

This single `Config` describes the whole experiment. Nothing is hidden elsewhere.

**Layers** (background): top-to-bottom horizontal layers; the last one has `bottom=None`.
For a homogeneous background just keep one layer.

**Targets** (buried bodies): each is either
- a **circle** -> give `center=[x, z]` and `radius` (z is negative, downward), or
- a **polygon** -> give `points=[[x, z], ...]`.

**Survey**: `scheme` is the electrode array - `dd` (dipole-dipole), `wa` (Wenner),
`slm` (Schlumberger), `wb`, `pd`, `pp`, `gr`.

**Noise**: one dataset is simulated and inverted per level (0.05 = 5 %).

**Inversion method**: `occam` (smooth) | `marquardt` (damping) | `blocky` (sharp) | `robust` (outlier-resistant).


In [ ]:
cfg = Config(
    # Background layers (top -> bottom; last one bottom=None) 
    layers=[
        Layer(bottom=-10.0, resistivity=50.0,  name="background"),
        Layer(bottom=None,  resistivity=200.0, name="substratum"),
    ],

    # Buried targets: circle (center+radius) OR polygon (points)
    targets=[
        Target(name="conductive", resistivity=10.0,  center=[-8, -3], radius=2.5),
        Target(name="resistive",  resistivity=100.0, center=[ 8, -3], radius=2.5),
        # polygon example:
        # Target(name="block", resistivity=300.0,
        #        points=[[-4, -4], [4, -4], [4, -8], [-4, -8]]),
    ],

    # Survey: array, number of electrodes, extent (m)
    survey=Survey(scheme="dd", n_electrodes=41, x_start=-20, x_end=20),

    # Noise levels to simulate & invert (fractions) 
    forward=Forward(noise_levels=[0.02, 0.05, 0.10]),

    # Inversion method: occam | marquardt | blocky | robust 
    inversion=Inversion(method="occam"),

    # Output + how deep to draw the figures (m)
    outdir=os.path.join(REPO_ROOT, "results"),
    datadir=os.path.join(REPO_ROOT, "data"),
    plot_depth=15.0,
)
print(cfg.describe())

ERTInv configuration
--------------------
  layers      : 2
  targets     : 2
  survey      : dd, 41 electrodes (-20 to 20 m)
  noise levels: ['2%', '5%', '10%']
  inversion   : occam (lam=None, z_weight=1.0)


## 3. Run

Simulates the data, inverts every noise level, and writes one PDF
(`results/ERTInv_report_<method>.pdf`) with four pages: true model, pseudosections,
recovered models, residual histograms. The last cell opens it.


In [5]:
results = run(cfg)
#print()
print(results.rms_table())

ERTInv | method = occam (Smooth (first-order) Occam-type inversion)
        noise levels = ['2%', '5%', '10%']



    2% noise : chi2 = 0.99 (RMS 1.00), model corr = 0.879


    5% noise : chi2 = 0.93 (RMS 0.97), model corr = 0.908
   10% noise : chi2 = 1.02 (RMS 1.01), model corr = 0.885

Done. Report -> c:\Users\dmadelis\Documents\git\ERTInv-main\results\ERTInv_report_occam.pdf
method             2%       5%      10%
---------------------------------------
occam            1.00     0.97     1.01


In [4]:
report = os.path.join(cfg.outdir, f'ERTInv_report_{cfg.inversion.method}.pdf')
print('Report:', report)
try:
    os.startfile(report)
except Exception:
    print('(open it manually from the path above)')

Report: c:\Users\dmadelis\Documents\git\ERTInv-main\results\ERTInv_report_occam.pdf


### How to read the numbers

`chi2` is the noise-weighted data misfit and `RMS = sqrt(chi2)`.
The target is **chi2 ~ 1**: the model fits the data to within the noise.
`chi2 >> 1` = under-fit; `chi2 << 1` = over-fit (chasing noise). The **model
correlation** (printed per level) says how much the recovered section resembles
the truth - only possible because this is synthetic.

*Optional:* to compare all four methods on one dataset, run in a new cell:
`compare_methods(cfg, methods=["occam","marquardt","blocky","robust"], noise_level=0.05)`
